Assigment 3
```
# COSC5437002 - Neural Networks and Deep Learning
# Prof. Syed Muhammad Danish
# Algoma University
# Department of Computer Science and Mathematics
# Brampton, Ontario, Canada
# Date: 8 jul 2025
```

# Final CNN:
- Added Data Augmentation
- Used BachNorm + ReLU + MaxPool + SGD
- Added new feature/layer (3 in total)
- Added scheduler
- Added Early Stopping



In [ ]:
import matplotlib.pyplot as plt
import torchvision
import numpy as np
from torchvision import transforms
import torch
from torchvision import datasets
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.init as init

In [ ]:
transform = transforms.Compose([ # Transform: flatten the image to a vector
    transforms.RandomHorizontalFlip(), # ADDED
    transforms.RandomCrop(32, padding=4), # ADDED
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),  # normalize to [-1, 1]
])

train_dataset = datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)
print(f"Total training samples: {len(train_dataset)}")
test_dataset = datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)
print(f"Total test samples: {len(test_dataset)}")

# ~> Setting batch size as 32
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

100%|██████████| 169M/169M [00:03<00:00, 48.6MB/s]


Total training samples: 50000
Total test samples: 10000
cuda


In [ ]:
class SimpleCNN(nn.Module): # Model definition
    def __init__(self):
        super(SimpleCNN, self).__init__()

        # Convolutional layers with BatchNorm
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # ADDED
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(256)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully connected layers
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(256 * 4 * 4, 1024) # CHANGED
        self.bn4 = nn.BatchNorm1d(1024)
        self.relu4 = nn.ReLU()
        self.fc2 = nn.Linear(1024, 100)  # CIFAR-100 has 100 classes

    def forward(self, x):
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
        x = self.pool3(self.relu3(self.bn3(self.conv3(x)))) # ADDED
        x = self.flatten(x)
        x = self.relu4(self.bn4(self.fc1(x))) # CHANGED
        x = self.fc2(x)

        return x

# Instantiate and send to device
model = SimpleCNN().to(device)

# ~> Loss: Cross Entropy is for classification models
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

# ADDED
from torch.optim.lr_scheduler import StepLR
scheduler = StepLR(optimizer, step_size=7, gamma=0.1)  # Learning rate decay

# Early Stopping Parameters
patience = 5 # How many epochs to wait after last time validation accuracy improved.
min_delta = 0.001 # Minimum change in the monitored quantity to qualify as an improvement.

best_accuracy = -1.0
epochs_no_improve = 0

In [ ]:
# Training loop with weight visualization
numEpoch = 20
for epoch in range(numEpoch):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    # Step the learning rate scheduler
    scheduler.step()

    train_accuracy = correct/total*100
    print(f"\nEpoch [{epoch+1}/{numEpoch}] Loss: {total_loss:.4f} Training Accuracy: {train_accuracy:.2f}%")

    # Evaluate on the test set for early stopping
    model.eval()
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            test_correct += (predicted == labels).sum().item()
            test_total += labels.size(0)

    current_test_accuracy = (test_correct / test_total) * 100
    print(f"Current Test Accuracy: {current_test_accuracy:.2f}%")

    # Check for early stopping
    if current_test_accuracy > best_accuracy + min_delta:
        best_accuracy = current_test_accuracy
        epochs_no_improve = 0
        # Optional: Save the best model
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"New best test accuracy: {best_accuracy:.2f}%. Model saved.")
    else:
        epochs_no_improve += 1
        print(f"No improvement for {epochs_no_improve} epochs.")
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs. Best Test Accuracy: {best_accuracy:.2f}%")
            break

# Load the best model if you saved it
# model.load_state_dict(torch.load('best_model.pth'))

print(f"\nFinal Test Accuracy after training (or early stopping): {best_accuracy:.2f}%")


Epoch [1/20] Loss: 5280.5956 Training Accuracy: 19.38%
Current Test Accuracy: 26.36%
New best test accuracy: 26.36%. Model saved.

Epoch [2/20] Loss: 4064.0581 Training Accuracy: 33.59%
Current Test Accuracy: 32.25%
New best test accuracy: 32.25%. Model saved.

Epoch [3/20] Loss: 3526.0578 Training Accuracy: 40.72%
Current Test Accuracy: 39.81%
New best test accuracy: 39.81%. Model saved.

Epoch [4/20] Loss: 3159.7431 Training Accuracy: 45.86%
Current Test Accuracy: 44.40%
New best test accuracy: 44.40%. Model saved.

Epoch [5/20] Loss: 2904.8120 Training Accuracy: 49.71%
Current Test Accuracy: 44.97%
New best test accuracy: 44.97%. Model saved.

Epoch [6/20] Loss: 2673.8869 Training Accuracy: 53.19%
Current Test Accuracy: 50.24%
New best test accuracy: 50.24%. Model saved.

Epoch [7/20] Loss: 2500.0048 Training Accuracy: 55.79%
Current Test Accuracy: 51.20%
New best test accuracy: 51.20%. Model saved.

Epoch [8/20] Loss: 2037.4853 Training Accuracy: 63.96%
Current Test Accuracy: 56.9